# K-sensitivity — inference-only

**Purpose.** Report the ladder's sensitivity to the evaluation window
$K_{\mathrm{eval}}$. Because training augments over $K \in 1\ldots 13$,
each trained checkpoint has already seen every context length; scoring
at a different $K$ is inference-only — the test set is cut at the new
$K$ and the same model produces predictions.

This notebook does **not** retrain anything. It writes new predictions
into `outputs/k{K}/<arm>/test_predictions.csv` so the study's canonical
results under `outputs/<arm>/` are untouched.

Sections:

1. Choose the sweep grid — $K \in \{2, 3, 5, 6\}$ by default.
2. Re-score each arm at each $K$.
3. Read back the per-$K$ predictions and score them.
4. Assemble the sensitivity table and figure.
5. Export.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kineret.config import paths
from kineret.config import data_config as C
from kineret.benchmark import ARMS, arm_label
from kineret.evaluation import (
    load_predictions, outcome_names_from,
    per_outcome_metrics, aggregate,
)
from kineret.multi_k import evaluate_at_k

OUT_ROOT = paths.OUTPUT_ROOT
K_VALUES = [2, 3, 5, 6]                        # canonical K = 4 is in outputs/
ARM_KEYS = ["logreg", "strats",                # skip the QA-only variants for
             "intervene_std", "intervene_kb"]  # a compact sensitivity table

print(f"Canonical K = {C.EVAL_CONTEXT_DAYS}. Sweeping over {K_VALUES}.")

## 2 — Re-score each arm at each K

Each call is inference-only and takes a fraction of the training run
(LogReg: seconds, ss-STraTS: ~1–2 min, INTERVenE: ~2–5 min on the
A5000). Total budget is roughly (4 arms × 4 Ks × ~3 min) = ~50 minutes.

The trained checkpoints under `checkpoints/` are reused unchanged —
`resume=True` is passed to each arm's `run()`.

In [ ]:
for K in K_VALUES:
    for arm in ARM_KEYS:
        target = os.path.join(OUT_ROOT, f"k{K}", arm, "test_predictions.csv")
        if os.path.exists(target):
            print(f"[skip] {arm_label(arm)} @ K={K}: predictions exist")
            continue
        evaluate_at_k(arm, K, output_root=OUT_ROOT, verbose=True)

## 3 — Score every (arm, K) combination

Support-weighted AUROC / AUPRC per (arm, K). Uses the ladder's own scorer
for direct comparability with the canonical $K=4$ table.

In [ ]:
rows = []

def _score(run_dir):
    preds = load_predictions(run_dir)
    outs = outcome_names_from(preds)
    lab = preds[[f"label_{o}" for o in outs]].to_numpy(dtype=float)
    prb = preds[[f"prob_{o}"  for o in outs]].to_numpy(dtype=float)
    per = per_outcome_metrics(lab, prb, outs)
    agg = aggregate(per)
    return agg[agg["average"] == "weighted"].iloc[0]

# Canonical K first (from outputs/<arm>/) so it appears as its own row.
for arm in ARM_KEYS:
    run_dir = os.path.join(OUT_ROOT, ARMS[arm]["run_dir"])
    if not os.path.exists(os.path.join(run_dir, "test_predictions.csv")):
        continue
    r = _score(run_dir)
    rows.append({"arm": arm_label(arm), "K": int(C.EVAL_CONTEXT_DAYS),
                 "AUROC": float(r["auroc"]), "AUPRC": float(r["auprc"]),
                 "F1":    float(r["best_f1"])})

# Then every off-K.
for K in K_VALUES:
    for arm in ARM_KEYS:
        run_dir = os.path.join(OUT_ROOT, f"k{K}", arm)
        if not os.path.exists(os.path.join(run_dir, "test_predictions.csv")):
            continue
        r = _score(run_dir)
        rows.append({"arm": arm_label(arm), "K": int(K),
                     "AUROC": float(r["auroc"]), "AUPRC": float(r["auprc"]),
                     "F1":    float(r["best_f1"])})

k_df = pd.DataFrame(rows).sort_values(["arm", "K"]).reset_index(drop=True)
k_df.round(3)

## 4 — Sensitivity figure — AUPRC vs K, per arm

In [ ]:
FIG_DIR = os.path.join(OUT_ROOT, "figures", "multi_k")
os.makedirs(FIG_DIR, exist_ok=True)

fig, ax = plt.subplots(figsize=(6.5, 3.8))
for arm, chunk in k_df.groupby("arm"):
    ax.plot(chunk["K"], chunk["AUPRC"], marker="o", label=arm, lw=1.4)
ax.set_xlabel("Evaluation window $K_{\\mathrm{eval}}$ (days)")
ax.set_ylabel("Support-weighted AUPRC")
ax.set_title("Ladder sensitivity to evaluation window (inference-only)")
ax.legend(fontsize=8)
ax.grid(True, lw=0.4, alpha=0.4)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "k_sensitivity.png"), dpi=200)
fig.savefig(os.path.join(FIG_DIR, "k_sensitivity.pdf"))
k_df.to_csv(os.path.join(FIG_DIR, "k_sensitivity.csv"), index=False)
fig

## Caveats

- **The head layout is fixed at the canonical $K = 4$** — the target list
  is support-filtered once, on train, at that window, so an outcome that
  clears $1\%$ prevalence at $K = 2$ but not at $K = 4$ (or vice versa)
  is NOT added or removed for this sweep. This is intentional: the point
  is to test whether the ladder ordering is robust to $K$, not to
  redecide which outcomes are scored.
- **Test cohort membership** is still filtered on $K_{\mathrm{eval}} \ge
  \text{shortest\_train\_window}$, matching the canonical run. Shorter
  admissions that were dropped at $K = 4$ are still dropped at $K = 2$
  under the same rule, so cohort composition is stable across the sweep.
- The QA-only variants (`_qa` arms) are omitted from the default sweep to
  keep the table compact. Add them to `ARM_KEYS` above to include them.